In [2]:
!pip install openai chromadb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 1.7 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openai]2m1/2 [openai]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [3]:
import os
from openai import OpenAI 
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [ ]:
load_dotenv()
api_key = ""

In [5]:
openai_client = OpenAI(api_key=api_key, base_url="https://aisuite.cirrascale.com/apis/v2")
chroma_client = chromadb.PersistentClient(path="./chroma_db")
COLLECTION_NAME = "baggage_policy"

In [6]:
embedding_function = embedding_functions.OpenAIEmbeddingFunction(
    api_key=api_key,
    model_name="BAAI/bge-large-en-v1.5",
    api_base="https://aisuite.cirrascale.com/apis/v2"
)

In [7]:
def index():
    baggage_rules = [
        "Emirates Airlines offers a generous baggage policy that varies based on route, fare type, and cabin class.",
        "For carry-on luggage, Economy passengers are allowed one piece weighing up to 7 kg with dimensions not exceeding 55 x 38 x 20 cm.",
        "First and Business Class passengers can bring two pieces of carry-on baggage: one briefcase and either a handbag or garment bag.",
        "Emirates uses either a 'weight concept' or 'piece concept' for checked baggage, depending on the route.",
        "Economy Special fares typically allow 20 kg of checked baggage, while Economy Flex Plus offers up to 35 kg.",
        "Each checked bag must not exceed 32 kg, with total dimensions not exceeding 300 cm (118 inches).",
        "Passengers traveling with infants can bring a carrycot or collapsible stroller on board or check it for free.",
        "Liquid items in carry-on luggage must be in containers of 100ml or less, placed in a single transparent, resealable 1-liter plastic bag.",
        "Emirates Skywards members may be eligible for additional baggage allowance on routes using the weight concept.",
        "For flights within the Americas and between the US and Europe, Emirates uses a 'piece concept' with different baggage allowances."
    ]

    collection = chroma_client.get_or_create_collection(
        name=COLLECTION_NAME,
        embedding_function=embedding_function,
        metadata={"hnsw:space": "cosine"}
    )
    
    if len(collection.get()['ids']) == 0:
        for idx, rule in enumerate(baggage_rules):
            collection.add(documents=[rule], ids=[f"baggage_rule_{idx}"])
        print(f"Stored {len(baggage_rules)} baggage rules in ChromaDB.")

    return collection

In [8]:
def retrieve(query: str):
    collection = chroma_client.get_collection(name=COLLECTION_NAME, embedding_function=embedding_function)
    results = collection.query(query_texts=[query], n_results=3)
    
    if results and results['documents']:
        return " ".join(results['documents'][0])
    return "No relevant baggage information found."    

In [17]:
def generate(query: str) -> str:
    context=retrieve(query)
    sys_prompt="""
    You are a helpful airline assistant. Answer the user query based on the context provided.
    If you don't find the answer within the context, say I don't know. Give accurate and precise answers.
    """
    usr_prompt=f"Query: {query}\nContext: {context}"
    
    response = openai_client.chat.completions.create(
        model="Llama-3.1-8B",
        messages=[
            {"role": "system", "content": sys_prompt},
            {"role": "user", "content": usr_prompt},
        ],
    )
    return response.choices[0].message.content

In [18]:
index()

Collection(name=baggage_policy)

In [19]:
query="What's the size limit for cabin baggage in economy class?"
answer=generate(query)
print(f"Q: {query}")
print(f"A: {answer}")

Q: What's the size limit for cabin baggage in economy class?
A: The size limit for cabin baggage in economy class is 55 x 38 x 20 cm.


In [12]:
query="What's the weight of checked baggage for Economy Flex?"
answer=generate(query)
print(f"Q: {query}")
print(f"A: {answer}")

Q: What's the weight of checked baggage for Economy Flex?
A: Based on the given context, the weight of checked baggage for Economy Flex is up to 35 kg.
